In [ ]:
import os
from pathlib import Path

PROJECT_ROOT = Path("../")
os.chdir(PROJECT_ROOT)

In [ ]:
from src.evaluation import evaluate_one, evaluate_env

In [ ]:
import torch
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

# Load base model
base_model = AutoModelForCausalLM.from_pretrained(
    "godninja/Affine-jeep_v6-5CG64fEwbCN6ysc3wVWfyTWjEKCCvtpjZ5dS5f43P4f3oXXY",  # Your base model
    torch_dtype="auto",
    device_map="auto"
)

# # Load adapter
model = PeftModel.from_pretrained(base_model, "outputs/sft_reasoning_trl/checkpoint-9000")

# Merge
merged_model = model.merge_and_unload()

# Save
# merged_model.save_pretrained("outputs/merged_model_8000")
tokenizer = AutoTokenizer.from_pretrained("godninja/Affine-jeep_v6-5CG64fEwbCN6ysc3wVWfyTWjEKCCvtpjZ5dS5f43P4f3oXXY")
# tokenizer.save_pretrained("outputs/merged_model_8000")

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = "affinier/affine-train-24"

model = AutoModelForCausalLM.from_pretrained(
    model_name, torch_dtype="auto", device_map="auto"
)

tokenizer = AutoTokenizer.from_pretrained(model_name)

In [ ]:
result = await evaluate_one(model, "arc-gen", task_id=400, tokenizer=tokenizer) # 400_000_000
print(result)

In [ ]:

result = await evaluate_one(model, "lgc-v2", task_id=300000010, tokenizer=tokenizer) # 400_000_000
print(result)


In [ ]:
import random

numbers = random.choices(range(400000000), k=5)
print(numbers)  # e.g., [2, 8, 8, 1, 5]

results = await evaluate_env(model, "lgc-v2", task_ids=numbers, tokenizer=tokenizer)
print(results)

In [ ]:
result = await evaluate_one(model, "print", task_id=1000, tokenizer=tokenizer) # 23000
print(result)

In [ ]:
numbers = random.choices(range(23300), k=5)
print(numbers)  # e.g., [2, 8, 8, 1, 5]

results = await evaluate_env(model, "print", task_ids=numbers, tokenizer=tokenizer)
print(results)

In [ ]:
result = await evaluate_one(model, "game", task_id=10_005_100, tokenizer=tokenizer) # 600_000_000
print(result)


In [ ]:
for i, msg in enumerate(result['rollout']):  # Show first 6 messages
    print(f"[{msg['role'].upper()}]")
    content = msg['content']#[:500] + "..." if len(msg['content']) > 500 else msg['content']
    reasoning_content = msg.get('reasoning_content', '')
    if msg['role'] == "assistant" and reasoning_content:
        print(f"THINK\n{reasoning_content}\n---\n")
    print(content)
    print("-" * 40)

In [ ]:
numbers = random.choices(range(600000000), k=5)
print(numbers)  # e.g., [2, 8, 8, 1, 5]

results = await evaluate_env(model, "game", task_ids=numbers, tokenizer=tokenizer)
print(results)

### Response Evaluation (Reward)

In [ ]:
from src.evaluation import reward_response

result = await reward_response(env="lgc-v2", task_id=783963231, response="129485763\n863927541\n475163298\n637214859\n254879136\n981356472\n316742985\n798531624\n542698317", ground_truth="2 + 2 = 4")
print(result)

## VLLM

In [ ]:
from vllm import LLM
from transformers import AutoTokenizer
from src.evaluation import evaluate_env_vllm, evaluate_one_vllm

import os
os.environ["VLLM_USE_V1"] = "0"
# Load vLLM model
model_name = "Qwen/Qwen3-4B"
llm = LLM(
    model_name,
    tensor_parallel_size=1,  # Adjust for multi-GPU
    gpu_memory_utilization=0.7,
    max_model_len=8192,
)
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Evaluate single task
result = await evaluate_one_vllm(
    llm=llm,
    env="lgc-v2",  # or "print", "game", "arc-gen"
    task_id=300000010,
    tokenizer=tokenizer,
    temperature=0.7,
    max_new_tokens=16384,
)
print(result)

# Evaluate multiple tasks
# results = await evaluate_env_vllm(
#     llm=llm,
#     env="lgc-v2",
#     task_ids=[1, 2, 3, 4, 5],
#     tokenizer=tokenizer,
# )
# print(f"Average score: {results['score']}")